# 01. 小売・EC売上データセット作成
経済産業省「商業動態統計」をベースにした業態別月次販売額の時系列データを生成します。
DataRobot AutoTSでの高精度モデリングに適した特徴量設計を含みます。

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import os

## 設定パラメータ

In [ ]:
START_YEAR = 2015
END_YEAR = 2025          # 学習データの最終年
ACTUALS_MONTHS = 3       # 予測期間 = 実績比較期間 (ヶ月)
STORE_TYPES = ["百貨店", "スーパー", "コンビニ", "ドラッグストア", "EC"]

# 出力ファイルパス
DATA_DIR = Path("../data")
TRAINING_CSV = DATA_DIR / "retail_sales_dataset.csv"       # 学習データ
SCORING_CSV  = DATA_DIR / "retail_sales_scoring.csv"        # スコアリング(予測入力)
ACTUALS_CSV  = DATA_DIR / "retail_sales_actuals.csv"        # 実績データ(グラフ比較用)

np.random.seed(42)

## 外部特徴量の生成
消費者信頼感指数、CPI、気温、失業率、祝日数などマクロ経済・環境データを生成します。

In [ ]:
# 日付範囲の生成
# 学習データ: 2015-01 〜 2025-12 (132ヶ月)
# 実績データ: 2026-01 〜 2026-03 (3ヶ月) ← 予測と比較するための「答え合わせ」用
# → 合計 135ヶ月分を一括生成し、後で分割する

train_end = f"{END_YEAR}-12-01"
actuals_end = pd.Timestamp(train_end) + pd.DateOffset(months=ACTUALS_MONTHS)

date_range_all = pd.date_range(start=f"{START_YEAR}-01-01", end=actuals_end, freq="MS")
n_months_all = len(date_range_all)
print(f"全期間: {date_range_all[0].strftime('%Y-%m')} 〜 {date_range_all[-1].strftime('%Y-%m')} ({n_months_all}ヶ月)")

# 学習期間の境界
TRAIN_CUTOFF = pd.Timestamp(train_end)
print(f"学習データ: ～ {TRAIN_CUTOFF.strftime('%Y-%m')}")
print(f"実績データ: {(TRAIN_CUTOFF + pd.DateOffset(months=1)).strftime('%Y-%m')} ～ {actuals_end.strftime('%Y-%m')}")

# 基本カラム (全期間)
months = np.array([d.month for d in date_range_all])
years = np.array([d.year for d in date_range_all])

# フラグ特徴量
is_bonus_month = np.isin(months, [6, 12])
is_golden_week = (months == 5)
is_year_end = np.isin(months, [11, 12])

# 消費者信頼感指数: 2015年~42, COVID(2020)で~35に低下, その後~37-39に回復
t = np.arange(n_months_all)
consumer_confidence = np.full(n_months_all, 42.0)
for i in range(n_months_all):
    y = years[i]
    if y < 2020:
        consumer_confidence[i] = 42.0 - (y - 2015) * 0.4
    elif y == 2020:
        consumer_confidence[i] = 35.0 + (months[i] - 1) * 0.3
    else:
        consumer_confidence[i] = 37.0 + min(y - 2021, 5) * 0.5
consumer_confidence += np.random.normal(0, 1.0, n_months_all)
consumer_confidence = np.round(consumer_confidence, 1)

# CPI: 2015年~98.5, 2020年まで~99-100, その後上昇し2025年~105-108
cpi = np.full(n_months_all, 98.5)
for i in range(n_months_all):
    y = years[i]
    m = months[i]
    if y <= 2020:
        cpi[i] = 98.5 + (y - 2015) * 0.3 + (m - 1) * 0.025
    else:
        cpi[i] = 100.0 + (y - 2020) * 1.5 + (m - 1) * 0.1
cpi += np.random.normal(0, 0.3, n_months_all)
cpi = np.round(cpi, 1)

# 東京の月別平均気温
tokyo_monthly_temp = {
    1: 5.2, 2: 5.7, 3: 8.7, 4: 13.9, 5: 18.2, 6: 21.4,
    7: 25.0, 8: 26.4, 9: 22.8, 10: 17.5, 11: 12.1, 12: 7.6
}
avg_temperature = np.array([tokyo_monthly_temp[m] for m in months])
avg_temperature += np.random.normal(0, 1.5, n_months_all)
avg_temperature = np.round(avg_temperature, 1)

# 失業率
unemployment_rate = np.full(n_months_all, 3.3)
for i in range(n_months_all):
    y = years[i]
    m = months[i]
    if y < 2020:
        unemployment_rate[i] = 3.3 - (y - 2015 + (m - 1) / 12) * 0.22
    elif y == 2020:
        if m <= 6:
            unemployment_rate[i] = 2.2 + (m / 6) * 0.8
        else:
            unemployment_rate[i] = 3.0 - ((m - 6) / 6) * 0.2
    else:
        unemployment_rate[i] = 2.8 - min(y - 2021, 5) * 0.06 - ((m - 1) / 12) * 0.06
unemployment_rate += np.random.normal(0, 0.1, n_months_all)
unemployment_rate = np.round(np.clip(unemployment_rate, 2.0, 4.0), 1)

# 祝日数
base_holidays = {1: 1, 2: 1, 3: 1, 4: 1, 5: 2, 6: 0, 7: 1, 8: 1, 9: 2, 10: 1, 11: 1, 12: 1}
num_holidays = np.array([base_holidays[m] for m in months])
holiday_noise = np.random.choice([-1, 0, 0, 0, 1], size=n_months_all)
num_holidays = np.clip(num_holidays + holiday_noise, 0, 3).astype(int)

# 外部特徴量DataFrame (全期間)
df_external = pd.DataFrame({
    "year_month": date_range_all,
    "month": months,
    "is_bonus_month": is_bonus_month,
    "is_golden_week": is_golden_week,
    "is_year_end": is_year_end,
    "consumer_confidence_index": consumer_confidence,
    "cpi": cpi,
    "avg_temperature": avg_temperature,
    "unemployment_rate": unemployment_rate,
    "num_holidays": num_holidays
})

print(f"外部特徴量テーブル: {df_external.shape}")
df_external.tail()

## 業態別売上データの生成
各業態ごとに、トレンド・季節性・イベント効果・ノイズを組み合わせてリアルな月次売上を生成します。

In [ ]:
def generate_sales(date_range, store_type):
    """業態別の月次売上（億円）を生成する関数"""
    n = len(date_range)
    months = np.array([d.month for d in date_range])
    years = np.array([d.year for d in date_range])
    t = np.arange(n)  # 月インデックス

    if store_type == "百貨店":
        base = 0.50
        trend = base * (-0.02 / 12) * t
        seasonality = np.where(months == 12, base * 0.30, 0)
        bonus = np.where(np.isin(months, [6, 12]), base * 0.15, 0)
        covid = np.where(years == 2020, -base * 0.20, 0)
        noise_std = base * 0.04

    elif store_type == "スーパー":
        base = 1.20
        trend = base * (0.01 / 12) * t
        seasonality = np.where(np.isin(months, [7, 8]), base * 0.08, 0)
        seasonality += np.where(months == 12, base * 0.10, 0)
        bonus = np.zeros(n)
        covid = np.where(years == 2020, base * 0.05, 0)
        noise_std = base * 0.03

    elif store_type == "コンビニ":
        base = 0.90
        trend = base * (0.02 / 12) * t
        seasonality = np.where(np.isin(months, [7, 8]), base * 0.10, 0)
        bonus = np.zeros(n)
        covid = np.where(years == 2020, -base * 0.08, 0)
        noise_std = base * 0.03

    elif store_type == "ドラッグストア":
        base = 0.55
        trend = base * (0.05 / 12) * t
        seasonality = np.where(np.isin(months, [2, 3, 4]), base * 0.08, 0)
        bonus = np.zeros(n)
        covid = np.where(years == 2020, base * 0.10, 0)
        noise_std = base * 0.03

    elif store_type == "EC":
        base = 0.15
        trend = base * ((1.15 ** (t / 12)) - 1)
        seasonality = np.where(np.isin(months, [11, 12]), base * 0.20 * (1.15 ** (t / 12)), 0)
        bonus = np.zeros(n)
        covid = np.where(np.isin(years, [2020, 2021]), base * 0.40 * (1.15 ** (t / 12)), 0)
        noise_std = base * 0.05 * (1.15 ** (t / 12))

    else:
        raise ValueError(f"Unknown store type: {store_type}")

    sales = base + trend + seasonality + bonus + covid + np.random.normal(0, noise_std, n)
    sales = np.maximum(sales, 0.01)
    return np.round(sales, 4)


# 全期間 (学習+実績) の売上を一括生成
all_records = []
for store_type in STORE_TYPES:
    sales = generate_sales(date_range_all, store_type)
    for i, d in enumerate(date_range_all):
        all_records.append({
            "year_month": d,
            "store_type": store_type,
            "sales_billion_yen": sales[i]
        })

df_sales = pd.DataFrame(all_records)
print(f"売上データ (全期間): {df_sales.shape}")
print(f"業態数: {df_sales['store_type'].nunique()}, 期間: {n_months_all}ヶ月")
print(f"合計レコード数: {len(df_sales)}")

## データの結合とプレビュー
売上データと外部特徴量をマージし、最終データセットを作成します。

In [ ]:
# 売上データと外部特徴量を結合
df = pd.merge(df_sales, df_external, on="year_month", how="left")

# ソート
df = df.sort_values(["year_month", "store_type"]).reset_index(drop=True)

print(f"最終データセット: {df.shape}")
df.head(10)

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ===== 1. 学習データ: ～2025-12 =====
df_train = df[df["year_month"] <= TRAIN_CUTOFF].copy()
df_train.to_csv(TRAINING_CSV, index=False)
print(f"[1] 学習データ: {TRAINING_CSV.name}")
print(f"    行数: {len(df_train)}, 期間: {df_train['year_month'].min()} ～ {df_train['year_month'].max()}")

# ===== 2. スコアリングデータ: 直近12ヶ月(実績) + 未来3ヶ月(target=NaN) =====
# 直近12ヶ月の実績 (特徴量導出ウィンドウ分)
recent_start = TRAIN_CUTOFF - pd.DateOffset(months=11)
df_recent = df_train[df_train["year_month"] >= recent_start].copy()

# 未来3ヶ月 (target列をNaNに)
df_future = df[df["year_month"] > TRAIN_CUTOFF].copy()
df_future["sales_billion_yen"] = np.nan  # 予測対象なので空にする

df_scoring = pd.concat([df_recent, df_future], ignore_index=True)
df_scoring = df_scoring.sort_values(["store_type", "year_month"]).reset_index(drop=True)
df_scoring.to_csv(SCORING_CSV, index=False)
print(f"\n[2] スコアリングデータ: {SCORING_CSV.name}")
print(f"    行数: {len(df_scoring)} (実績{len(df_recent)}行 + 未来{len(df_future)}行)")
print(f"    期間: {df_scoring['year_month'].min()} ～ {df_scoring['year_month'].max()}")

# ===== 3. 実績データ: 2026-01～2026-03 (予測と同期間、グラフ比較用) =====
df_actuals = df[df["year_month"] > TRAIN_CUTOFF].copy()
df_actuals.to_csv(ACTUALS_CSV, index=False)
print(f"\n[3] 実績データ (グラフ比較用): {ACTUALS_CSV.name}")
print(f"    行数: {len(df_actuals)}, 期間: {df_actuals['year_month'].min()} ～ {df_actuals['year_month'].max()}")

print(f"\n✅ 3つのCSVファイルを {DATA_DIR} に保存しました。")

In [ ]:
print("=== データセット概要 (全期間) ===")
print(f"全期間: {len(df)} 行, {len(df.columns)} カラム")
print(f"  学習: {len(df_train)} 行 (～{TRAIN_CUTOFF.strftime('%Y-%m')})")
print(f"  スコアリング: {len(df_scoring)} 行 (直近12ヶ月+未来{ACTUALS_MONTHS}ヶ月)")
print(f"  実績: {len(df_actuals)} 行 (未来{ACTUALS_MONTHS}ヶ月の答え)")

print("\n=== 業態別の統計 (学習データ) ===")
print(df_train.groupby("store_type")["sales_billion_yen"].describe().round(2))

print("\n=== 実績データプレビュー ===")
print(df_actuals[["year_month", "store_type", "sales_billion_yen"]].to_string(index=False))

## DataRobot AI Catalogへのアップロード
生成したデータセットをDataRobotのAI Catalogにアップロードします。

In [ ]:
# DataRobot AI Catalogへのアップロード (3つのCSV)
try:
    from dotenv import load_dotenv
    load_dotenv("../.env")

    import datarobot as dr
    dr.Client()

    # 1. 学習データ
    train_ds = dr.Dataset.create_from_file(file_path=str(TRAINING_CSV))
    print(f"[1] 学習データ アップロード完了: {train_ds.id}")

    # 2. スコアリングデータ
    scoring_ds = dr.Dataset.create_from_file(file_path=str(SCORING_CSV))
    print(f"[2] スコアリングデータ アップロード完了: {scoring_ds.id}")

    # 3. 実績データ
    actuals_ds = dr.Dataset.create_from_file(file_path=str(ACTUALS_CSV))
    print(f"[3] 実績データ アップロード完了: {actuals_ds.id}")

    print(f"\n{'='*60}")
    print(f"以下を .env に追記してください:")
    print(f"{'='*60}")
    print(f"TRAINING_DATASET_ID={train_ds.id}")
    print(f"SCORING_DATASET_ID={scoring_ds.id}")
    print(f"ACTUALS_DATASET_ID={actuals_ds.id}")
    print(f"{'='*60}")
except Exception as e:
    print(f"DataRobotへのアップロードをスキップしました: {e}")
    print("手動でアップロードする場合は、DataRobot UIからCSVをアップロードしてください。")